# Experimentul 11 · verificarea analizei finite-shot și a costului cuantic

**Analiză suplimentară** a Exp. 11 (`exp11_quantum_readout_optimization.ipynb`). **Nu reantrenăm nimic** și nu modificăm rezultatele oficiale: refolosim scorurile exacte salvate (`results/result_exp11/scores/`, matrici de densitate, `default.mixed`, zgomot oficial) și ponderile salvate. Toate fișierele noi au prefixul `finite_shot_`; testul nu este accesat.

## Ce verificăm
1. **Matematic:** pentru O ∈ {Z₀, Z₁, Z₂, Z₃, Z₀Z₁} (valori proprii ±1), s = (1 − ⟨O⟩)/2 = P(O = −1), deci K ~ Binomial(N, s), ŝ = K/N este estimatorul cu N shots.
2. **Independent, cu PennyLane:** eșantionare directă pe `default.mixed` cu modelul oficial de zgomot, pe tranzacții reale din validation, comparată statistic cu (A) valoarea exactă și (B) eșantionarea binomială.
3. **Monte Carlo:** N = 1 024 / 4 096 / 10 000 / 32 768 / 100 000, toate observabilele, seeds și p, **50 de repetări**; AUPRC, AUROC, Spearman față de scorurile exacte, metrici la trei praguri (A: pragul exact original; B: max-F1 recalculat pe scorurile cu shots; C: prag ales pe o jumătate stratificată a validării, evaluat pe cealaltă).
4. **Separabilitate și cost:** separarea claselor vs eroarea de eșantionare, distribuții și cuantile, shots necesare, ledger pe bugete de shots.

**Variații separate:** variația Monte Carlo (între repetări, același seed) și variația între cele 5 seeds (modele diferite) se raportează separat; media pe seeds se calculează din mediile Monte Carlo per seed (n = 5 observații independente, nu 250).

## 1. Configurație, date, scoruri salvate

Seeds: fiecare configurație (observabilă, seed, p, N) are propriul generator `np.random.default_rng([2026, i_obs, seed, i_p, N])`, deci rezultatele nu depind de ordinea execuției și nicio configurație nu reutilizează realizările alteia. Împărțirile stratificate pentru pragul C sunt aceleași pentru toate configurațiile (repetarea k folosește împărțirea k), ca să fie comparabile pereche.

In [ ]:
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import torch
from IPython.display import display
from scipy.stats import chi2, ks_2samp, spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
from src.quantum_readout import (NATIVE_GATES, N_QUBITS, OBSERVABLES, STAR_PAIRS, AngleEncoder, ansatz, choose_threshold,
                                 evaluate, make_feature_map, make_model, make_vqc, noisy_qnode, predict, readout, undersample)

OBS = list(OBSERVABLES)
BASELINE = "Z0"
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, 0.001, 0.01, 0.05]
SHOT_LEVELS = [1024, 4096, 10_000, 32_768, 100_000]
N_REPS = 50
BASE_SEED = 2026
SPLIT_SEED = 42
P_FOCUS = 0.05
PRIORITY = [("Z0", 0.05, n) for n in (1024, 10_000, 100_000)] + [("Z0Z1", 0.05, n) for n in (1024, 10_000, 100_000)]
THRESHOLDS = {"A": "prag exact original", "B": "max-F1 recalculat pe shots", "C": "prag pe jumătate, evaluat pe cealaltă"}
CONF = ["precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN"]
REP_METRICS = ["AUPRC", "AUROC", "Spearman"] + [f"{k} [{t}]" for t in THRESHOLDS for k in CONF]

# verificarea PennyLane directă
VERIF_SEED = 0
VERIF_P = [0.0, 0.05]
VERIF_SHOTS = 1024
VERIF_REPS = 200
EXPVAL_REPS = 100
Z_LIMIT = 4.0                 # |z| al mediei față de valoarea exactă
VAR_BAND = (chi2.ppf(0.0005, VERIF_REPS - 1) / (VERIF_REPS - 1), chi2.ppf(0.9995, VERIF_REPS - 1) / (VERIF_REPS - 1))
EXACT_TOL = 1e-12

RESULTS_DIR = REPO_ROOT / "results" / "result_exp11"
SCORES_DIR = RESULTS_DIR / "scores"
REPS_DIR = RESULTS_DIR / "finite_shot_reps"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP9_DIR = REPO_ROOT / "results" / "result_exp9"
REPS_DIR.mkdir(exist_ok=True)

OBS_LABELS = {"Z0": "⟨Z₀⟩", "Z1": "⟨Z₁⟩", "Z2": "⟨Z₂⟩", "Z3": "⟨Z₃⟩", "Z0Z1": "⟨Z₀Z₁⟩"}
OBS_COLORS = {"Z0": "#2a78d6", "Z1": "#eb6834", "Z2": "#1baf7a", "Z3": "#eda100", "Z0Z1": "#e87ba4"}
OBS_MARKERS = {"Z0": "o", "Z1": "s", "Z2": "D", "Z3": "^", "Z0Z1": "P"}
SEED_DOT_COLOR, LEGIT_COLOR, FRAUD_COLOR = "#8a8f98", "#8a8f98", "#e34948"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
torch.set_default_dtype(torch.float64)
np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)

DATA_PATH, SPLIT_PATH = REPO_ROOT / "data" / "creditcard.csv", REPO_ROOT / "split_indices.npz"
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
split = np.load(SPLIT_PATH)
config_exp11 = json.loads((RESULTS_DIR / "config.json").read_text(encoding="utf-8"))
split_sha = hashlib.sha256(b"".join(np.ascontiguousarray(split[k]).tobytes() for k in ["train", "val", "test"])).hexdigest()
assert split_sha == config_exp11["split_sha256"], "alt split decât Exp. 11"
X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
X_train, y_train = X_all.loc[split["train"]], y_all[split["train"]]
X_val, y_val = X_all.loc[split["val"]], y_all[split["val"]]
del X_all, y_all, df   # testul nu este folosit
assert (len(y_val), int(y_val.sum())) == (56962, 99)
POS = y_val == 1
N_POS, N_NEG = int(POS.sum()), int((~POS).sum())

SCORES = {(o, s, p): np.load(SCORES_DIR / f"{o}_seed{s}_p{p}.npz")["val"] for o in OBS for s in SEEDS for p in P_ALL}
official = pd.read_csv(RESULTS_DIR / "metrics_per_seed.csv")
off = official.set_index(["observabilă", "seed", "p"])
align_err = 0.0
for (o, s, p), sc in SCORES.items():   # aliniere scoruri ↔ etichete: AUPRC și pragul oficial se recalculează identic
    assert sc.dtype == np.float64 and sc.shape == y_val.shape and np.isfinite(sc).all()
    align_err = max(align_err, abs(average_precision_score(y_val, sc) - off.loc[(o, s, p), "AUPRC"]),
                    abs(choose_threshold(y_val, sc) - off.loc[(o, s, p), "threshold"]))
assert align_err < EXACT_TOL, align_err
print(f"✅ {len(SCORES)} seturi de scoruri exacte încărcate; AUPRC și pragurile oficiale recalculate identic (eroare {align_err:.1e}); "
      f"scoruri în [{min(s.min() for s in SCORES.values()):.6f}, {max(s.max() for s in SCORES.values()):.6f}]")

## 2. Verificarea matematică și a implementării inițiale

**Formula.** Pentru O = Z_k: în baza computațională, rezultatul b_k = 1 ⇔ valoarea proprie −1, deci ⟨Z_k⟩ = P(b_k = 0) − P(b_k = 1) = 1 − 2 P(b_k = 1) și s = (1 − ⟨Z_k⟩)/2 = P(b_k = 1). Pentru O = Z₀Z₁: valoarea proprie a lui |b⟩ este (−1)^(b₀ ⊕ b₁), deci ⟨Z₀Z₁⟩ = 1 − 2 P(b₀ ⊕ b₁ = 1) și s = P(paritate impară). În ambele cazuri, fiecare shot este o variabilă Bernoulli(s), shots-urile sunt i.i.d. (aceeași stare ρ, preparată de N ori), deci K = numărul de rezultate „−1” ~ Binomial(N, s), iar ŝ = K/N = (1 − Ô)/2 cu Ô = media celor N valori ±1: estimatorul obișnuit al lui ⟨O⟩, transformat liniar. E[ŝ] = s, Var[ŝ] = s(1 − s)/N. Fiecare tranzacție este o execuție separată, deci eșantioanele sunt independente între tranzacții.

**Implementarea inițială** (celula „Sensibilitatea la numărul de shots” din notebook-ul Exp. 11), verificată punct cu punct:

| Punct | Constatare |
|---|---|
| Probabilitatea eșantionată | `np.clip(SCORES[o, seed, p], 0, 1)` = s = P(O = −1). Corect pentru toate cele 5 observabile. |
| Convenția semnului | ŝ = K/N cu K = numărul de rezultate −1, aceeași orientare ca scorul exact (fraudă ⇔ scor mare). Corect. |
| Limitele [0, 1] | toate scorurile salvate sunt strict în (0, 1) (intervalul este afișat în celula de mai sus); `clip` nu modifică nimic. |
| Tipuri | float64 → `binomial` întoarce int64, împărțit la N în float64. Corect. |
| Independența | un singur `default_rng(42)` folosit secvențial: fiecare apel `binomial(N, s)` cu vectorul s trage eșantioane independente per tranzacție și per repetare; nicio realizare nu este refolosită. Corect, dar rezultatele depindeau de ordinea buclei (aici: un generator per configurație). |
| Alinierea scoruri ↔ etichete | scorurile salvate reproduc exact AUPRC și pragurile oficiale (verificat mai sus). Corect. |
| Pragul | **recalculat** max-F1 pe scorurile cu shots (varianta B). Pragul exact (A) nu fusese raportat. |
| Repetări | **10**: prea puține pentru deviația standard Monte Carlo; aici 50. Lipseau AUROC, Spearman, MCC, TN/FN. |

Verificăm numeric formula și tipurile, apoi faptul că valorile lui `evaluate` coincid cu implementarea rapidă folosită în bucla Monte Carlo.

In [ ]:
def fast_conf(pos, n_pos, n_neg, s, thr):
    # aceleași mărimi ca evaluate() (sklearn), calculate direct; verificat mai jos
    pred = s >= thr
    tp = int(np.count_nonzero(pred & pos))
    fp = int(np.count_nonzero(pred)) - tp
    fn, tn = n_pos - tp, n_neg - fp
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / n_pos
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denom if denom else 0.0
    return [precision, recall, f1, mcc, tp, fp, fn, tn]


rng_check = np.random.default_rng(0)
s_test = SCORES["Z0Z1", 0, 0.05]
for n in [1, 10, 1024]:
    draws = rng_check.binomial(n, np.full(200_000, 0.3)) / n   # media și varianța binomiale
    assert abs(draws.mean() - 0.3) < 5 * math.sqrt(0.21 / n / 200_000) and abs(draws.var() / (0.21 / n) - 1) < 0.02
s_hat = rng_check.binomial(1024, s_test) / 1024
assert s_hat.dtype == np.float64 and s_hat.min() >= 0 and s_hat.max() <= 1
for s_eval in [s_test, s_hat]:
    for thr in [choose_threshold(y_val, s_eval), 0.5, np.quantile(s_eval, 0.999)]:
        ref = evaluate(y_val, s_eval, thr)
        mine = fast_conf(POS, N_POS, N_NEG, s_eval, thr)
        assert np.allclose(mine, [ref[k] for k in CONF], atol=1e-12), (mine, ref)
print("✅ binomial: medie și varianță corecte; fast_conf = evaluate (precision, recall, F1, MCC, TP, FP, FN, TN)")

## 3. Verificare independentă cu PennyLane (eșantionare directă cu shots finite)

`noisy_qnode` oficial folosește `diff_method="backprop"`, care **nu acceptă shots finite** pe `default.mixed` (PennyLane 0.45.1 ridică `QuantumFunctionError`). Construim aceeași funcție cu `diff_method=None`: același dispozitiv `default.mixed`, același `qml.noise.insert(DepolarizingChannel, p, position="all")`, aceeași descompunere în `NATIVE_GATES`. Modelul de zgomot nu se schimbă; verificăm că varianta exactă (`shots=None`) a acestui QNode dă aceleași probabilități ca `noisy_qnode` oficial.

Pentru seed 0 și fiecare observabilă (modelul ei antrenat), alegem 6 tranzacții reale din validation cu scoruri diferite la p = 0.05 (fraudele cu scor maxim, median, minim; legitimele cu scor maxim, median, minim) și le evaluăm la p = 0 și p = 0.05:
- **A:** valoarea exactă (`noisy_qnode` oficial); trebuie să fie egală cu scorul salvat (aliniere).
- **C:** PennyLane, `qml.sample(wires=[0, 1, 2, 3])` cu 200 × 1 024 shots într-o execuție, împărțite în 200 de repetări de 1 024; ŝ = fracția de rezultate cu b_k = 1 (Z_k) sau cu b₀ ⊕ b₁ = 1 (Z₀Z₁).
- **B:** 200 de eșantioane Binomial(1 024, s).

Criterii: |z| < 4 pentru media PennyLane față de s; varianța PennyLane / (s(1 − s)/N) în banda χ² 99.9% (n = 200); test KS între C și B (raportat). În plus, pentru Z₀ și Z₀Z₁ la p = 0.05, calea `qml.expval(O)` cu `shots=1024` (100 de execuții separate) pe 2 tranzacții.

In [ ]:
def shot_qnode(fn, p, shots):
    # = noisy_qnode oficial, cu diff_method=None ca să accepte shots finite
    dev = qml.noise.insert(qml.device("default.mixed", wires=N_QUBITS), qml.DepolarizingChannel, p, position="all")
    qnode = qml.transforms.decompose(qml.QNode(fn, dev, diff_method=None), gate_set=NATIVE_GATES)
    return qnode if shots is None else qml.set_shots(qnode, shots=shots)


FEATURE_MAP = make_feature_map(STAR_PAIRS)


def sample_circuit(x, weights):
    FEATURE_MAP(x)
    ansatz(weights)
    return qml.sample(wires=range(N_QUBITS))


def probs_circuit(x, weights):
    FEATURE_MAP(x)
    ansatz(weights)
    return qml.probs(wires=range(N_QUBITS))


def estimator_from_bits(o, bits):
    if o == "Z0Z1":
        return bits[..., 0] ^ bits[..., 1]   # paritate impară ⇔ valoarea proprie −1 a lui Z0 Z1
    return bits[..., int(o[1])]


def load_weights(o, seed):
    path = EXP9_DIR / "weights" / "star" / f"seed_{seed}.pt" if o == BASELINE else RESULTS_DIR / "weights" / o / f"seed_{seed}.pt"
    return torch.load(path, weights_only=True)["weights"]


X_s, y_s = undersample(X_train, y_train, VERIF_SEED)
Z_val0 = AngleEncoder(N_QUBITS, VERIF_SEED).fit(X_s).transform(X_val)
dev4 = qml.device("default.qubit", wires=N_QUBITS)
verif_rows, expval_rows = [], []
rng_verif = np.random.default_rng([BASE_SEED, 999])
t_verif = time.perf_counter()
for o in OBS:
    w = load_weights(o, VERIF_SEED)
    clean = predict(Z_val0, w, make_model(qml.QNode(make_vqc(STAR_PAIRS, o), dev4, interface="torch", diff_method="backprop")))
    assert np.abs(clean - SCORES[o, VERIF_SEED, 0.0]).max() < EXACT_TOL, f"{o}: unghiurile / ponderile nu reproduc scorurile salvate"
    ref = SCORES[o, VERIF_SEED, P_FOCUS]
    f_idx, l_idx = np.flatnonzero(POS), np.flatnonzero(~POS)
    picks = {"fraudă, scor maxim": f_idx[np.argmax(ref[f_idx])], "fraudă, scor median": f_idx[np.argsort(ref[f_idx])[len(f_idx) // 2]],
             "fraudă, scor minim": f_idx[np.argmin(ref[f_idx])], "legitimă, scor maxim": l_idx[np.argmax(ref[l_idx])],
             "legitimă, scor median": l_idx[np.argsort(ref[l_idx])[len(l_idx) // 2]], "legitimă, scor minim": l_idx[np.argmin(ref[l_idx])]}
    for p in VERIF_P:
        sampler = shot_qnode(sample_circuit, p, VERIF_SHOTS * VERIF_REPS)
        exact_probs_q = shot_qnode(probs_circuit, p, None)
        for name, i in picks.items():
            x = torch.tensor(Z_val0[i])
            s_exact = float(readout(noisy_qnode(make_vqc(STAR_PAIRS, o), p)(x, w)))
            assert abs(s_exact - SCORES[o, VERIF_SEED, p][i]) < EXACT_TOL, "valoarea exactă ≠ scorul salvat (aliniere)"
            P = np.asarray(exact_probs_q(x, w))
            official_probs = np.asarray(noisy_qnode(probs_circuit, p)(x, w))
            assert np.abs(P - official_probs).max() < EXACT_TOL, "QNode-ul cu shots are alt model de zgomot"
            bits = np.asarray(sampler(x, w)).reshape(VERIF_REPS, VERIF_SHOTS, N_QUBITS)
            pl = estimator_from_bits(o, bits).mean(axis=1)
            if o == "Z0Z1":   # paritatea = (1 − media produsului valorilor ±1)/2, exact pe aceleași shots
                z0, z1 = 1 - 2 * bits[..., 0], 1 - 2 * bits[..., 1]
                assert np.allclose(pl, (1 - (z0 * z1).mean(axis=1)) / 2, atol=1e-15)
            binom = rng_verif.binomial(VERIF_SHOTS, s_exact, size=VERIF_REPS) / VERIF_SHOTS
            var_th = s_exact * (1 - s_exact) / VERIF_SHOTS
            z = (pl.mean() - s_exact) / math.sqrt(var_th / VERIF_REPS)
            ratio = pl.var(ddof=1) / var_th
            row = {"observabilă": o, "p": p, "tranzacție": name, "index validation": int(i), "clasă": int(y_val[i]), "s exact (A)": s_exact,
                   "medie PennyLane (C)": pl.mean(), "medie binomial (B)": binom.mean(), "var teoretică s(1−s)/N": var_th,
                   "var PennyLane / teoretică": ratio, "var binomial / teoretică": binom.var(ddof=1) / var_th,
                   "z medie PennyLane": z, "KS p-value (C vs B)": ks_2samp(pl, binom).pvalue,
                   "trecut": abs(z) < Z_LIMIT and VAR_BAND[0] < ratio < VAR_BAND[1]}
            verif_rows.append(row)
            if p == P_FOCUS and o in (BASELINE, "Z0Z1") and name in ("fraudă, scor maxim", "legitimă, scor median"):
                ev = shot_qnode(make_vqc(STAR_PAIRS, o), p, VERIF_SHOTS)
                vals = np.array([float(readout(ev(x, w))) for _ in range(EXPVAL_REPS)])
                expval_rows.append({"observabilă": o, "p": p, "tranzacție": name, "s exact": s_exact, "medie expval cu shots": vals.mean(),
                                    "z": (vals.mean() - s_exact) / math.sqrt(var_th / EXPVAL_REPS), "var / teoretică": vals.var(ddof=1) / var_th,
                                    "trecut": abs((vals.mean() - s_exact) / math.sqrt(var_th / EXPVAL_REPS)) < Z_LIMIT})
verif = pd.DataFrame(verif_rows)
expval_check = pd.DataFrame(expval_rows)
pd.concat([verif.assign(cale="qml.sample, 200 × 1024 shots"), expval_check.assign(cale="qml.expval, 100 × 1024 shots")]).to_csv(
    RESULTS_DIR / "finite_shot_verification.csv", index=False)
display(verif.round(5))
display(expval_check.round(5))
print(f"banda varianței (χ², 99.9%, n = {VERIF_REPS}): {VAR_BAND[0]:.3f} – {VAR_BAND[1]:.3f};  timp {time.perf_counter() - t_verif:.0f} s")
assert verif["trecut"].all() and expval_check["trecut"].all(), "eșantionarea PennyLane nu concordă cu modelul binomial"
print(f"✅ {len(verif)} + {len(expval_check)} verificări PennyLane trecute; KS p-value minim {verif['KS p-value (C vs B)'].min():.3f}")

## 4. Monte Carlo finite-shot (50 de repetări, toate configurațiile)

Pentru fiecare (observabilă, p, N) și fiecare seed: 50 de realizări ŝ = Binomial(N, s)/N pe toate cele 56 962 de tranzacții (scorurile exacte salvate). Per realizare: AUPRC, AUROC, Spearman ρ(ŝ, s), și precision / recall / F1 / MCC / TP / FP / FN / TN la pragurile A (exact original, din `metrics_per_seed.csv`), B (max-F1 recalculat pe ŝ, același validation: optimist) și C (max-F1 pe o jumătate stratificată a validării, evaluat pe cealaltă jumătate; repetarea k folosește împărțirea k, ~49–50 de fraude în fiecare jumătate). Referința C pe scorurile exacte folosește aceleași 50 de împărțiri.

Realizările se salvează per (observabilă, p, N) în `finite_shot_reps/` (5 seeds × 50 de repetări × 27 de metrici); o rulare întreruptă reia de unde a rămas. Configurațiile prioritare (p = 0.05, Z₀ și Z₀Z₁, 1 024 / 10 000 / 100 000) rulează primele.

In [ ]:
SPLITS = [train_test_split(np.arange(len(y_val)), test_size=0.5, stratify=y_val, random_state=1000 + k) for k in range(N_REPS)]
SPLIT_POS = [(POS[a], POS[b]) for a, b in SPLITS]


def rep_metrics(s_hat, s_exact, thr_exact, k):
    out = [average_precision_score(y_val, s_hat), roc_auc_score(y_val, s_hat), spearmanr(s_exact, s_hat).statistic]
    out += fast_conf(POS, N_POS, N_NEG, s_hat, thr_exact)
    out += fast_conf(POS, N_POS, N_NEG, s_hat, choose_threshold(y_val, s_hat))
    a, b = SPLITS[k]
    pa, pb = SPLIT_POS[k]
    out += fast_conf(pb, int(pb.sum()), int((~pb).sum()), s_hat[b], choose_threshold(pa.astype(int), s_hat[a]))
    return out


def reps_path(o, p, n):
    return REPS_DIR / f"{o}_p{p}_N{n}.npz"


todo = PRIORITY + [(o, p, n) for p in [0.05, 0.01, 0.001, 0.0] for o in OBS for n in SHOT_LEVELS if (o, p, n) not in PRIORITY]
assert len(todo) == len(set(todo)) == len(OBS) * len(P_ALL) * len(SHOT_LEVELS)
t_all = time.perf_counter()
for o, p, n in todo:
    if reps_path(o, p, n).exists():
        continue
    t0 = time.perf_counter()
    arr = np.empty((len(SEEDS), N_REPS, len(REP_METRICS)))
    for si, seed in enumerate(SEEDS):
        s = np.clip(SCORES[o, seed, p], 0.0, 1.0)
        thr_exact = off.loc[(o, seed, p), "threshold"]
        rng = np.random.default_rng([BASE_SEED, OBS.index(o), seed, P_ALL.index(p), n])
        for k in range(N_REPS):
            arr[si, k] = rep_metrics(rng.binomial(n, s) / n, s, thr_exact, k)
    np.savez_compressed(reps_path(o, p, n), reps=arr, metrics=np.array(REP_METRICS), seeds=np.array(SEEDS), seconds=time.perf_counter() - t0)
    print(f"{o:<5} p = {p:<5} N = {n:>6}: AUPRC {arr[:, :, 0].mean():.4f} (MC std mediu {arr[:, :, 0].std(axis=1, ddof=1).mean():.4f})  "
          f"({time.perf_counter() - t0:.0f} s)")
print(f"Monte Carlo complet ({(time.perf_counter() - t_all) / 60:.1f} min în această rulare)")

REPS = {}
for o in OBS:
    for p in P_ALL:
        for n in SHOT_LEVELS:
            z = np.load(reps_path(o, p, n))
            assert list(z["metrics"]) == REP_METRICS and z["reps"].shape == (len(SEEDS), N_REPS, len(REP_METRICS)) and np.isfinite(z["reps"]).all()
            REPS[o, p, n] = z["reps"]

### Reproducerea analizei inițiale

Analiza inițială (10 repetări, prag B) raportase la p = 0.05, 1 024 shots: Z₀ 0.520, Z₀Z₁ 0.188 (medii pe seeds). Cu 50 de repetări și alte realizări aleatoare, mediile trebuie să coincidă în limita erorii Monte Carlo.

In [ ]:
initial = pd.read_csv(RESULTS_DIR / "shot_noise_sensitivity.csv")
repro_rows = []
for o in OBS:
    for n in [1024, 10_000, 100_000]:
        new = REPS[o, P_FOCUS, n][:, :, REP_METRICS.index("AUPRC")]
        old = initial[(initial["observabilă"] == o) & (initial["p"] == P_FOCUS) & (initial["shots"] == n)].set_index("seed").loc[SEEDS]
        se = np.sqrt(new.var(axis=1, ddof=1) / N_REPS + old["AUPRC shots (std repetări)"].to_numpy() ** 2 / 10)   # eroarea standard a diferenței, per seed
        diff = new.mean(axis=1) - old["AUPRC shots (medie repetări)"].to_numpy()
        repro_rows.append({"observabilă": o, "shots": n, "AUPRC inițial (10 rep.)": old["AUPRC shots (medie repetări)"].mean(),
                           "AUPRC acum (50 rep.)": new.mean(), "max |z| per seed": float(np.max(np.abs(diff / se)))})
repro = pd.DataFrame(repro_rows)
display(repro.round(4))
assert (repro["max |z| per seed"] < 4).all(), "analiza inițială nu se reproduce"
print("✅ analiza inițială se reproduce în limita erorii Monte Carlo (|z| < 4 per seed)")

## 5. Statistici: variație Monte Carlo vs variație între seeds

`finite_shot_metrics.csv`: per (observabilă, seed, p, N, metrică): valoarea exactă (pentru metricile care au una), media, std (ddof = 1), min și max pe cele 50 de repetări. `finite_shot_summary.csv`: per (observabilă, p, N, metrică): media pe seeds a mediilor MC, std între seeds (ddof = 1) a mediilor MC, media std-urilor MC din interiorul seed-urilor.

In [ ]:
EXACT_COLS = {"AUPRC": "AUPRC", "AUROC": "AUROC", **{f"{k} [B]": k for k in CONF}}   # protocolul oficial = prag max-F1 pe scorurile exacte


def exact_value(o, seed, p, metric):
    if metric == "Spearman":
        return 1.0
    if metric in EXACT_COLS:
        return float(off.loc[(o, seed, p), EXACT_COLS[metric]])
    if metric.endswith("[A]"):   # pragul exact aplicat scorurilor exacte = protocolul oficial
        return float(off.loc[(o, seed, p), metric[:-4]])
    return np.nan


# referința C pe scorurile exacte (aceleași 50 de împărțiri)
C_EXACT = {}
for o in OBS:
    for seed in SEEDS:
        for p in P_ALL:
            s = SCORES[o, seed, p]
            vals = []
            for k, (a, b) in enumerate(SPLITS):
                pa, pb = SPLIT_POS[k]
                vals.append(fast_conf(pb, int(pb.sum()), int((~pb).sum()), s[b], choose_threshold(pa.astype(int), s[a])))
            C_EXACT[o, seed, p] = np.mean(vals, axis=0)

rows = []
for (o, p, n), arr in REPS.items():
    for si, seed in enumerate(SEEDS):
        for mi, m in enumerate(REP_METRICS):
            v = arr[si, :, mi]
            ex = C_EXACT[o, seed, p][CONF.index(m[:-4])] if m.endswith("[C]") else exact_value(o, seed, p, m)
            rows.append({"observabilă": o, "seed": seed, "p": p, "shots": n, "metrică": m, "exact": ex,
                         "MC medie": v.mean(), "MC std": v.std(ddof=1), "MC min": v.min(), "MC max": v.max()})
fs_metrics = pd.DataFrame(rows)
fs_metrics.to_csv(RESULTS_DIR / "finite_shot_metrics.csv", index=False)

g = fs_metrics.groupby(["observabilă", "p", "shots", "metrică"])
fs_summary = pd.DataFrame({"exact: medie pe seeds": g["exact"].mean(), "exact: std între seeds": g["exact"].std(ddof=1),
                           "shots: medie pe seeds (din mediile MC)": g["MC medie"].mean(),
                           "shots: std între seeds (ale mediilor MC)": g["MC medie"].std(ddof=1),
                           "shots: std MC medie în interiorul seed-ului": g["MC std"].mean(),
                           "shots: min MC (toate seeds)": g["MC min"].min(), "shots: max MC (toate seeds)": g["MC max"].max()}).reset_index()
fs_summary.to_csv(RESULTS_DIR / "finite_shot_summary.csv", index=False)


def summ(o, p, n, metric, col="shots: medie pe seeds (din mediile MC)"):
    return fs_summary[(fs_summary["observabilă"] == o) & (fs_summary["p"] == p) & (fs_summary["shots"] == n) & (fs_summary["metrică"] == metric)][col].iloc[0]


def per_seed_mc(o, p, n, metric):
    return REPS[o, p, n][:, :, REP_METRICS.index(metric)].mean(axis=1)


view = []
for o in OBS:
    for n in SHOT_LEVELS:
        view.append({"observabilă": o, "shots": n, **{f"{m}": f"{summ(o, P_FOCUS, n, m):.4f} ± {summ(o, P_FOCUS, n, m, 'shots: std între seeds (ale mediilor MC)'):.4f} "
                                                        f"(MC {summ(o, P_FOCUS, n, m, 'shots: std MC medie în interiorul seed-ului'):.4f})"
                                                     for m in ["AUPRC", "AUROC", "Spearman", "recall [B]", "F1 [B]", "MCC [B]"]},
                     "FP [B]": f"{summ(o, P_FOCUS, n, 'FP [B]'):.1f}"})
    view.append({"observabilă": o, "shots": "exact", "AUPRC": f"{summ(o, P_FOCUS, 1024, 'AUPRC', 'exact: medie pe seeds'):.4f}",
                 "recall [B]": f"{summ(o, P_FOCUS, 1024, 'recall [B]', 'exact: medie pe seeds'):.4f}"})
print("p = 0.05: medie pe seeds ± std între seeds (MC = std medie între repetări, în interiorul unui seed)")
display(pd.DataFrame(view).set_index(["observabilă", "shots"]).fillna(""))

## 6. Comparația pereche Z₀Z₁ − Z₀ (și celelalte observabile − Z₀)

Per seed: Δ = media MC (observabila nouă) − media MC (Z₀); apoi medie ± std pe cele 5 seeds și numărul de seeds cu Δ > 0. În plus, P_MC = probabilitatea ca o realizare a observabilei noi să depășească o realizare a lui Z₀, în același seed (50 × 50 de perechi; realizările sunt independente), mediată pe seeds.

In [ ]:
paired_rows = []
for o in [x for x in OBS if x != BASELINE]:
    for p in P_ALL:
        for n in ["exact", *SHOT_LEVELS]:
            for m in ["AUPRC", "AUROC", "recall [B]", "F1 [B]", "FP [B]", "recall [C]", "F1 [C]"]:
                if n == "exact":
                    if m.endswith("[C]"):
                        d = np.array([C_EXACT[o, s, p][CONF.index(m[:-4])] - C_EXACT[BASELINE, s, p][CONF.index(m[:-4])] for s in SEEDS])
                    else:
                        d = np.array([exact_value(o, s, p, m) - exact_value(BASELINE, s, p, m) for s in SEEDS])
                    p_mc = np.nan
                else:
                    a, b = REPS[o, p, n][:, :, REP_METRICS.index(m)], REPS[BASELINE, p, n][:, :, REP_METRICS.index(m)]
                    d = a.mean(axis=1) - b.mean(axis=1)
                    p_mc = float(np.mean([(a[si][:, None] > b[si][None, :]).mean() for si in range(len(SEEDS))]))
                better = (d < 0) if m.startswith("FP") else (d > 0)
                paired_rows.append({"comparație": f"{o} − Z0", "p": p, "shots": n, "metrică": m, "Δ medie": d.mean(), "Δ std între seeds": d.std(ddof=1),
                                    "Δ min seed": d.min(), "Δ max seed": d.max(), "seeds cu observabila nouă mai bună": int(better.sum()),
                                    "P_MC(realizare nouă > realizare Z0), medie pe seeds": p_mc})
fs_paired = pd.DataFrame(paired_rows)
fs_paired.to_csv(RESULTS_DIR / "finite_shot_paired_comparison.csv", index=False)
show = fs_paired[(fs_paired["comparație"] == "Z0Z1 − Z0") & (fs_paired["metrică"] == "AUPRC") & (fs_paired["p"].isin([0.01, 0.05]))]
display(show.drop(columns=["comparație", "metrică"]).set_index(["p", "shots"]).round(4))

## 7. Analiza pragurilor: A (exact original) vs B (recalculat pe shots) vs C (jumătate / jumătate)

Pentru a separa **pierderea de informație** de **pragul ales**: B este cel mai bun prag max-F1 posibil pe scorurile cu shots (oracol optimist). Dacă F1 [B] cu shots < F1 exact, pierderea vine din clasament (informație pierdută), nu din prag: niciun prag nu o recuperează. Diferența F1 [B] − F1 [A] este partea care se datorează pragului (A rămâne la valoarea exactă, deși scorurile cu shots au altă scară). C este estimarea onestă (prag ales pe date independente de cele de evaluare), comparată cu C pe scorurile exacte.

In [ ]:
thr_rows = []
for o in OBS:
    for p in P_ALL:
        ex_f1 = summ(o, p, 1024, "F1 [B]", "exact: medie pe seeds")
        ex_rec = summ(o, p, 1024, "recall [B]", "exact: medie pe seeds")
        ex_fp = summ(o, p, 1024, "FP [B]", "exact: medie pe seeds")
        exC = np.mean([C_EXACT[o, s, p] for s in SEEDS], axis=0)
        for n in SHOT_LEVELS:
            row = {"observabilă": o, "p": p, "shots": n, "exact: F1 oficial": ex_f1, "exact: recall oficial": ex_rec, "exact: FP oficial": ex_fp,
                   "exact: F1 [C]": exC[CONF.index("F1")], "exact: recall [C]": exC[CONF.index("recall")]}
            for t in THRESHOLDS:
                for k in ["precision", "recall", "F1", "MCC", "TP", "FP"]:
                    row[f"{k} [{t}]"] = summ(o, p, n, f"{k} [{t}]")
                row[f"F1 [{t}] std între seeds"] = summ(o, p, n, f"F1 [{t}]", "shots: std între seeds (ale mediilor MC)")
            row["pierdere F1 din informație (exact − B)"] = ex_f1 - row["F1 [B]"]
            row["câștig F1 din recalibrarea pragului (B − A)"] = row["F1 [B]"] - row["F1 [A]"]
            row["pierdere F1 onestă (exact C − shots C)"] = exC[CONF.index("F1")] - row["F1 [C]"]
            thr_rows.append(row)
fs_thr = pd.DataFrame(thr_rows)
fs_thr.to_csv(RESULTS_DIR / "finite_shot_threshold_analysis.csv", index=False)
cols = ["shots", "exact: recall oficial", "recall [A]", "recall [B]", "recall [C]", "FP [A]", "FP [B]", "exact: F1 oficial", "F1 [A]", "F1 [B]", "F1 [C]",
        "pierdere F1 din informație (exact − B)", "câștig F1 din recalibrarea pragului (B − A)"]
for o in [BASELINE, "Z0Z1"]:
    print(f"{o}, p = 0.05")
    display(fs_thr[(fs_thr["observabilă"] == o) & (fs_thr["p"] == P_FOCUS)][cols].set_index("shots").round(3))

## 8. Separabilitate, distribuții și shots necesare

Per (observabilă, seed, p), pe scorurile exacte: media și cuantilele pe clase, diferența medie fraudă − legitim, d′ = diferență / √((σ²_f + σ²_l)/2), coeficientul de suprapunere (OVL) al histogramelor celor două clase, distanța dintre cuantila 99% a legitimelor și mediana fraudelor, factorul de contracție (din Exp. 11). Per N: deviația standard a estimatorului √(s(1 − s)/N), mediată pe tranzacții, și raportul semnal / zgomot = diferența medie / std shot. N_egal = 0.25 / σ²_legit: numărul de shots la care zgomotul de măsurare (≈ 0.5/√N) egalează dispersia intrinsecă a scorurilor legitime.

**Shots necesare:** N₉₅ = cel mai mic N (interpolare liniară în log N pe grila MC) la care AUPRC medie (pe seeds) ajunge la 95% din AUPRC exactă; N_0.01 = la fel, pentru AUPRC exactă − 0.01.

In [ ]:
def ovl(a, b, bins):
    ha, _ = np.histogram(a, bins=bins, density=True)
    hb, _ = np.histogram(b, bins=bins, density=True)
    return float(np.sum(np.minimum(ha, hb) * np.diff(bins)))


QS = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
sep_rows = []
for o in OBS:
    for seed in SEEDS:
        for p in P_ALL:
            s = SCORES[o, seed, p]
            f, l = s[POS], s[~POS]
            bins = np.linspace(s.min(), s.max(), 201)
            row = {"observabilă": o, "seed": seed, "p": p, "medie fraude": f.mean(), "medie legitime": l.mean(), "diferență medie": f.mean() - l.mean(),
                   "std fraude": f.std(ddof=1), "std legitime": l.std(ddof=1),
                   "d′": (f.mean() - l.mean()) / math.sqrt((f.var(ddof=1) + l.var(ddof=1)) / 2), "OVL (histograme, exact)": ovl(f, l, bins),
                   "mediana fraude − cuantila 99% legitime": np.median(f) - np.quantile(l, 0.99), "lățimea intervalului de scor": s.max() - s.min(),
                   "factor de contracție |⟨O⟩|": float(off.loc[(o, seed, p), "factor de contracție |⟨O⟩|"]),
                   "N_egal = 0.25 / σ²_legit": 0.25 / l.var(ddof=1)}
            row.update({f"fraude q{int(q * 100)}": np.quantile(f, q) for q in QS})
            row.update({f"legitime q{int(q * 100)}": np.quantile(l, q) for q in QS})
            for n in SHOT_LEVELS:
                sd = np.sqrt(s * (1 - s) / n).mean()
                row[f"std shot N={n}"] = sd
                row[f"SNR N={n} (diferență / std shot)"] = (f.mean() - l.mean()) / sd
            sep_rows.append(row)
separability = pd.DataFrame(sep_rows)


def shots_needed(o, p, target):
    x = np.log10(SHOT_LEVELS)
    y = np.array([summ(o, p, n, "AUPRC") for n in SHOT_LEVELS])
    if y[0] >= target:
        return f"≤ {SHOT_LEVELS[0]}"
    for i in range(1, len(y)):
        if y[i] >= target:
            return int(round(10 ** (x[i - 1] + (target - y[i - 1]) / (y[i] - y[i - 1]) * (x[i] - x[i - 1]))))
    return f"> {SHOT_LEVELS[-1]}"


need_rows = []
for o in OBS:
    for p in P_ALL:
        ex = summ(o, p, 1024, "AUPRC", "exact: medie pe seeds")
        sel = separability[(separability["observabilă"] == o) & (separability["p"] == p)]
        need_rows.append({"observabilă": o, "p": p, "AUPRC exactă": ex, "N₉₅ (95% din exact)": shots_needed(o, p, 0.95 * ex),
                          "N_0.01 (exact − 0.01)": shots_needed(o, p, ex - 0.01), "diferență medie fraudă − legitim": sel["diferență medie"].mean(),
                          "std legitime": sel["std legitime"].mean(), "d′": sel["d′"].mean(), "OVL": sel["OVL (histograme, exact)"].mean(),
                          "lățimea intervalului de scor": sel["lățimea intervalului de scor"].mean(),
                          "contracție |⟨O⟩|": sel["factor de contracție |⟨O⟩|"].mean(), "N_egal (medie seeds)": sel["N_egal = 0.25 / σ²_legit"].mean(),
                          "SNR la 1 024": sel["SNR N=1024 (diferență / std shot)"].mean(), "SNR la 100 000": sel["SNR N=100000 (diferență / std shot)"].mean()})
needs = pd.DataFrame(need_rows)
separability.merge(needs[["observabilă", "p", "N₉₅ (95% din exact)", "N_0.01 (exact − 0.01)"]], on=["observabilă", "p"]).to_csv(
    RESULTS_DIR / "finite_shot_separability.csv", index=False)
display(needs[needs["p"].isin([0.01, 0.05])].set_index(["p", "observabilă"]).round(4))

## 9. Cost ledger pe bugete de shots (scenarii hardware; ledger-ul oficial rămâne neschimbat)

Execuțiile de circuit per seed sunt cele din `cost_ledger.json` al Exp. 11 (antrenare cu parameter shift măsurat cu `qml.Tracker`; validare după fiecare epocă; checkpoint; validare cu zgomot la 3 niveluri de p). Shots totale = execuții × N. Z₀Z₁ folosește **aceleași** shots ca Z₀ per execuție (paritatea și cei doi biți vin din aceeași măsurare în baza Z). Separăm costul antrenării (gradient) de cel al validării / inferenței și dăm costul de inferență per tranzacție.

In [ ]:
ledger_exact = json.loads((RESULTS_DIR / "cost_ledger.json").read_text(encoding="utf-8"))
fs_ledger = {"note": "hardware scenarios derived from the Exp. 11 execution counts; the official exact-simulation ledger (cost_ledger.json) is unchanged; "
                     "Z0Z1 uses the same shots per execution as Z0 (both bits from one computational-basis measurement)",
             "shot_levels": SHOT_LEVELS, "reference_budget_shots_per_execution": 1024, "observables": {}}
ledger_rows = []
for o in OBS:
    ex = ledger_exact["observables"][o]["executions_per_seed"]
    train = ex["train_forward_plus_parameter_shift_tracker"]
    val_epochs, val_ckpt, val_noisy = ex["validation_after_each_epoch_p0"], ex["validation_p0_selected_checkpoint"], ex["validation_noisy_p0.001_0.01_0.05"]
    fs_ledger["observables"][o] = {"executions_per_seed": {"training_parameter_shift_tracker": train, "validation_each_epoch": val_epochs,
                                                           "validation_checkpoint": val_ckpt, "validation_noisy_3_levels": val_noisy},
                                   "executions_per_example_gradient_tracker": ledger_exact["observables"][o]["executions_per_example_tracker"],
                                   "per_shot_level": {}}
    for n in SHOT_LEVELS:
        entry = {"shots_per_execution": n, "inference_shots_per_transaction": n,
                 "training_shots_per_seed": train * n, "validation_and_inference_shots_per_seed": (val_epochs + val_ckpt + val_noisy) * n,
                 "total_shots_per_seed": (train + val_epochs + val_ckpt + val_noisy) * n,
                 "total_shots_5_seeds": (train + val_epochs + val_ckpt + val_noisy) * n * len(SEEDS),
                 "inference_shots_full_validation_one_noise_level": len(y_val) * n, "factor_vs_1024": n / 1024}
        fs_ledger["observables"][o]["per_shot_level"][str(n)] = entry
        ledger_rows.append({"observabilă": o, **entry})
(RESULTS_DIR / "finite_shot_cost_ledger.json").write_text(json.dumps(fs_ledger, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame(ledger_rows).set_index(["observabilă", "shots_per_execution"]).map(lambda v: f"{v:,.3g}" if isinstance(v, float) else f"{v:,}"))

## 10. Grafice

In [ ]:
def exact_mean(o, p, metric="AUPRC"):
    return summ(o, p, 1024, metric, "exact: medie pe seeds")


# fs1. AUPRC vs shots (log), p = 0.01 și 0.05
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, p in zip(axes, [0.01, 0.05]):
    for o in OBS:
        m = [summ(o, p, n, "AUPRC") for n in SHOT_LEVELS]
        sd = [summ(o, p, n, "AUPRC", "shots: std între seeds (ale mediilor MC)") for n in SHOT_LEVELS]
        ax.errorbar(SHOT_LEVELS, m, yerr=sd, color=OBS_COLORS[o], marker=OBS_MARKERS[o], markersize=7, linewidth=2, capsize=3, label=OBS_LABELS[o])
        ax.axhline(exact_mean(o, p), color=OBS_COLORS[o], linestyle=":", linewidth=1.2, alpha=0.9)
    ax.set_xscale("log")
    ax.set_xticks(SHOT_LEVELS, [f"{n:,}".replace(",", " ") for n in SHOT_LEVELS])
    ax.minorticks_off()
    ax.set_xlabel("shots per execuție (log)")
    ax.set_title(f"p = {p}")
axes[0].set_ylabel("AUPRC (validation), medie ± std între seeds")
axes[1].legend(fontsize=9, loc="lower right")
fig.suptitle("fs1 · AUPRC vs shots · punctat: AUPRC exactă (protocol oficial) · 50 de repetări MC per seed", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "fs1_auprc_vs_shots.png", dpi=130)
plt.show()

# fs2. recall vs shots, p = 0.05: prag B (plin), A (punctat), C (întrerupt)
fig, ax = plt.subplots(figsize=(9, 5))
for o in OBS:
    for t, ls, lab in [("B", "-", True), ("A", ":", False), ("C", "--", False)]:
        ax.plot(SHOT_LEVELS, [summ(o, P_FOCUS, n, f"recall [{t}]") for n in SHOT_LEVELS], color=OBS_COLORS[o], linestyle=ls,
                marker=OBS_MARKERS[o] if t == "B" else None, markersize=7, linewidth=2 if t == "B" else 1.3, label=OBS_LABELS[o] if lab else None)
    ax.scatter([400_000], [exact_mean(o, P_FOCUS, "recall [B]")], color=OBS_COLORS[o], marker=OBS_MARKERS[o], s=60, edgecolor="white", zorder=4)
ax.plot([], [], color="0.4", linestyle="-", label="prag B: max-F1 recalculat pe shots")
ax.plot([], [], color="0.4", linestyle=":", label="prag A: pragul exact original")
ax.plot([], [], color="0.4", linestyle="--", label="prag C: ales pe ½, evaluat pe ½")
ax.set_xscale("log")
ax.set_xticks([*SHOT_LEVELS, 400_000], [*[f"{n:,}".replace(",", " ") for n in SHOT_LEVELS], "exact"])
ax.minorticks_off()
ax.set_xlabel("shots per execuție (log)")
ax.set_ylabel("recall (medie pe seeds)")
ax.set_title("fs2 · Recall vs shots la p = 0.05, trei praguri")
ax.legend(fontsize=8, loc="lower right", ncol=2)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "fs2_recall_vs_shots_p0.05.png", dpi=130)
plt.show()

# fs3. Z0 vs Z0Z1 la p = 0.05: per seed, banda = ± std MC
fig, ax = plt.subplots(figsize=(9, 5))
for o in [BASELINE, "Z0Z1"]:
    for si, seed in enumerate(SEEDS):
        means = np.array([REPS[o, P_FOCUS, n][si, :, 0].mean() for n in SHOT_LEVELS])
        sds = np.array([REPS[o, P_FOCUS, n][si, :, 0].std(ddof=1) for n in SHOT_LEVELS])
        ax.fill_between(SHOT_LEVELS, means - sds, means + sds, color=OBS_COLORS[o], alpha=0.12, linewidth=0)
        ax.plot(SHOT_LEVELS, means, color=OBS_COLORS[o], linewidth=1.2, alpha=0.8, marker=OBS_MARKERS[o], markersize=4,
                label=f"{OBS_LABELS[o]} (câte o linie per seed)" if si == 0 else None)
        ax.scatter([400_000], [off.loc[(o, seed, P_FOCUS), "AUPRC"]], color=OBS_COLORS[o], marker=OBS_MARKERS[o], s=30, edgecolor="white", zorder=4)
ax.set_xscale("log")
ax.set_xticks([*SHOT_LEVELS, 400_000], [*[f"{n:,}".replace(",", " ") for n in SHOT_LEVELS], "exact"])
ax.minorticks_off()
ax.set_xlabel("shots per execuție (log)")
ax.set_ylabel("AUPRC la p = 0.05")
ax.set_title("fs3 · Z₀ vs Z₀Z₁ la p = 0.05 · per seed, banda = ± std Monte Carlo (50 rep.)")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "fs3_z0_vs_z0z1_p0.05.png", dpi=130)
plt.show()

# fs4. distribuțiile scorurilor (seed 0, p = 0.05): exact și o realizare cu 1 024 shots
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
rng_plot = np.random.default_rng([BASE_SEED, 4])
for row, o in enumerate([BASELINE, "Z0Z1"]):
    s = SCORES[o, 0, P_FOCUS]
    for col, (title, vals) in enumerate([("exact (protocol oficial)", s), ("o realizare cu 1 024 shots", rng_plot.binomial(1024, s) / 1024)]):
        ax = axes[row, col]
        bins = np.linspace(min(vals.min(), s.min()), max(vals.max(), s.max()), 80)
        ax.hist(vals[~POS], bins=bins, density=True, color=LEGIT_COLOR, alpha=0.55, label=f"legitime (n = {N_NEG:,})".replace(",", " "))
        ax.hist(vals[POS], bins=bins, density=True, color=FRAUD_COLOR, alpha=0.55, label=f"fraude (n = {N_POS})")
        for q in [0.5, 0.99]:
            ax.axvline(np.quantile(vals[~POS], q), color="0.3", linestyle=":", linewidth=1)
        ax.axvline(np.median(vals[POS]), color=FRAUD_COLOR, linestyle="--", linewidth=1.2)
        ax.set_title(f"{OBS_LABELS[o]} · {title} · AUPRC {average_precision_score(y_val, vals):.3f}", fontsize=10)
        ax.set_xlabel("scor s = (1 − ⟨O⟩)/2")
        ax.legend(fontsize=8, loc="upper right")
    axes[row, 0].set_ylabel("densitate")
fig.suptitle("fs4 · Distribuțiile scorurilor, seed 0, p = 0.05 · punctat: mediana și cuantila 99% a legitimelor · întrerupt: mediana fraudelor", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "fs4_score_distributions_p0.05.png", dpi=130)
plt.show()

# fs5. Δ AUPRC (Z0Z1 − Z0) vs shots, p = 0.01 și 0.05
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=False)
xs = [*SHOT_LEVELS, 400_000]
for ax, p in zip(axes, [0.01, 0.05]):
    for si, seed in enumerate(SEEDS):
        d = [REPS["Z0Z1", p, n][si, :, 0].mean() - REPS[BASELINE, p, n][si, :, 0].mean() for n in SHOT_LEVELS]
        d.append(off.loc[("Z0Z1", seed, p), "AUPRC"] - off.loc[(BASELINE, seed, p), "AUPRC"])
        ax.plot(xs, d, color=SEED_DOT_COLOR, marker="o", markersize=4, linewidth=1, alpha=0.8)
        ax.annotate(f"s{seed}", (xs[-1], d[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    sel = fs_paired[(fs_paired["comparație"] == "Z0Z1 − Z0") & (fs_paired["metrică"] == "AUPRC") & (fs_paired["p"] == p)].set_index("shots")
    mean_d = [sel.loc[n, "Δ medie"] for n in SHOT_LEVELS] + [sel.loc["exact", "Δ medie"]]
    ax.plot(xs, mean_d, color=OBS_COLORS["Z0Z1"], marker=OBS_MARKERS["Z0Z1"], markersize=8, linewidth=2.5, label="medie pe 5 seeds")
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks(xs, [*[f"{n:,}".replace(",", " ") for n in SHOT_LEVELS], "exact"])
    ax.minorticks_off()
    ax.set_xlabel("shots per execuție (log)")
    ax.set_title(f"Δ AUPRC (Z₀Z₁ − Z₀), p = {p}")
    ax.legend(fontsize=9, loc="lower right")
axes[0].set_ylabel("Δ AUPRC (> 0: Z₀Z₁ mai bun)")
fig.suptitle("fs5 · Diferența pereche Z₀Z₁ − Z₀ vs bugetul de shots (gri: seeds, din mediile MC)", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "fs5_delta_auprc_z0z1_vs_z0.png", dpi=130)
plt.show()
print("grafice salvate în", PLOTS_DIR.relative_to(REPO_ROOT))